# A_model — XGBoost Feature E (팀 공통 Group CV)

**목표:** 최신 `src/features.py`의 ① 변이 유형, ② 아미노산 치환, ③ 유전자 절단 플래그와 이진 변이·변이 수를 조합하여 XGBoost `max_depth=3` 및 비교 모델의 Macro F1을 평가합니다.

- 팀원 `src` 파일은 수정하지 않습니다.
- 데이터의 원본 행 순서를 유지합니다.
- OOF는 팀 공통 Fold의 validation 행에만 채웁니다.
- 예측 저장은 선택한 최종 모델에 대해서만 실행합니다.
- `src/features.py` 최신 버전의 함수 시그니처를 확인해야 합니다.

**장치:** XGBoost CUDA 사용 가능 여부를 실제 소규모 학습으로 자동 검사합니다. CUDA 사용 불가 시 CPU로 전환합니다.


In [1]:
from pathlib import Path
import sys, inspect, json, warnings
import numpy as np
import pandas as pd
from sklearn.base import clone
from xgboost import XGBClassifier

# 실행 위치가 저장소 루트 또는 그 하위인 경우 자동으로 src를 찾습니다.
start = Path.cwd().resolve()
ROOT = next((p for p in (start, *start.parents) if (p / 'src' / 'config.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError(f'저장소 루트(src/config.py)를 찾을 수 없습니다. 실행 위치: {start}')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import DATA_DIR, SEED, set_seed
from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import binarize_mutations
from src.validation import get_folds, run_cv
from src.evaluation import macro_f1, per_class_f1, log_result, save_predictions
import src.features as features

set_seed()
print('ROOT:', ROOT)
print('DATA_DIR:', DATA_DIR)
print('XGBoost:', __import__('xgboost').__version__)
# GPU 자동 탐지: 실제 XGBoost CUDA 학습을 소량 실행해 확인합니다.
def select_xgb_device():
    try:
        probe = XGBClassifier(
            n_estimators=1, max_depth=1, tree_method='hist',
            device='cuda', objective='binary:logistic',
            eval_metric='logloss', n_jobs=1, verbosity=0
        )
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            probe.fit(np.array([[0.], [1.], [2.], [3.]], dtype=np.float32),
                      np.array([0, 0, 1, 1]))
        config = json.loads(probe.get_booster().save_config())
        actual = config.get('learner', {}).get('generic_param', {}).get('device', '')
        if str(actual).startswith('cuda'):
            print(f'XGBoost GPU 확인: {actual}')
            return 'cuda'
        print(f'XGBoost가 GPU 대신 {actual or "CPU"}로 실행되어 CPU를 사용합니다.')
    except Exception as exc:
        print(f'XGBoost GPU 확인 실패 ({type(exc).__name__}: {exc}); CPU로 전환합니다.')
    return 'cpu'

XGB_DEVICE = select_xgb_device()
print('선택한 XGBoost 장치:', XGB_DEVICE)


ROOT: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype
DATA_DIR: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\data
XGBoost: 3.4.1
XGBoost GPU 확인: cuda:0
선택한 XGBoost 장치: cuda


In [2]:
# 최신 dev의 공통 피처 함수가 있는지 확인 (없으면 학습하지 않습니다).
required = ['mutation_count', 'mutation_type_counts', 'amino_acid_pair_counts', 'gene_truncation_flags']
missing = [name for name in required if not callable(getattr(features, name, None))]
if missing:
    raise RuntimeError('src/features.py에 최신 피처 함수가 없습니다: ' + ', '.join(missing) + '\n시윤님 PR 병합 후 최신 dev를 받아 주세요.')
for name in required:
    fn = getattr(features, name)
    print(name, inspect.signature(fn))

mutation_count (X: pandas.DataFrame) -> pandas.Series
mutation_type_counts (X)
amino_acid_pair_counts (X: pandas.DataFrame) -> pandas.DataFrame
gene_truncation_flags (X: pandas.DataFrame) -> pandas.DataFrame


In [3]:
train, test = load_data(DATA_DIR)
X_raw, y_raw = split_xy(train)
y, le = encode_target(y_raw)
X_test_raw = test[X_raw.columns].copy()
assert X_raw.columns.equals(X_test_raw.columns)
assert X_raw.index.equals(pd.RangeIndex(len(X_raw)))
assert X_test_raw.index.equals(pd.RangeIndex(len(X_test_raw)))
folds = get_folds(y, X=X_raw, classes=le.classes_)
assert len(folds) == 5
print('train:', X_raw.shape, 'test:', X_test_raw.shape, 'classes:', len(le.classes_))

train: (6201, 4384) test: (2546, 4384) classes: 26


In [4]:
# 함수는 팀 코드 그대로 호출합니다. 모든 피처는 행별로 계산해야 합니다.
def feature_frame(result, index, prefix):
    if isinstance(result, pd.Series):
        result = result.to_frame()
    if not isinstance(result, pd.DataFrame):
        raise TypeError(f'{prefix}: 반환값은 Series 또는 DataFrame이어야 합니다')
    if len(result) != len(index) or not result.index.equals(index):
        raise ValueError(f'{prefix}: 행 인덱스 또는 길이가 원본과 다릅니다')
    out = result.copy()
    out.columns = [f'{prefix}__{c}' for c in out.columns]
    return out

def make_E(X):
    binary = feature_frame(binarize_mutations(X), X.index, 'binary')
    count = feature_frame(np.log1p(features.mutation_count(X)), X.index, 'log_count')
    types = feature_frame(np.log1p(features.mutation_type_counts(X)), X.index, 'log_type')
    pairs = feature_frame(np.log1p(features.amino_acid_pair_counts(X)), X.index, 'log_pair')
    trunc = feature_frame(features.gene_truncation_flags(X), X.index, 'trunc')
    parts = [binary, count, types, pairs, trunc]
    if any(p.columns.has_duplicates for p in parts):
        raise ValueError('피처 내 중복 열 이름이 있습니다')
    result = pd.concat(parts, axis=1)
    if result.columns.has_duplicates or result.isna().any().any():
        raise ValueError('중복 열 또는 결측 피처가 있습니다')
    if not np.isfinite(result.to_numpy(dtype=np.float64)).all():
        raise ValueError('무한대/NaN 피처가 있습니다')
    return result.astype(np.float32)

X_E = make_E(X_raw)
X_test_E = make_E(X_test_raw)
assert X_E.columns.equals(X_test_E.columns), 'train/test 피처 열 또는 순서 불일치'
assert len(X_E) == len(y) and len(X_test_E) == len(test)
print('E 피처:', X_E.shape, X_test_E.shape)
print('참고: 기대 열 수 9174는 팀 최신 구현 기준이며, 실제 열 수를 확인하세요.')
if X_E.shape[1] != 9174:
    raise ValueError(f'팀 E 실험의 9174개 피처와 다릅니다: {X_E.shape[1]}개. 최신 함수의 열 정의를 확인하세요.')

E 피처: (6201, 9174) (2546, 9174)
참고: 기대 열 수 9174는 팀 최신 구현 기준이며, 실제 열 수를 확인하세요.


In [5]:
# 팀 Fold가 모든 행을 한 번씩 검증하는지 확인
seen = np.zeros(len(y), dtype=int)
for i, (tr, va) in enumerate(folds, 1):
    assert len(np.intersect1d(tr, va)) == 0
    seen[va] += 1
    print(f'Fold {i}: train={len(tr)}, val={len(va)}')
assert np.all(seen == 1)

# 2개 설정을 동일한 Fold에서 비교합니다.
common = dict(n_estimators=250, learning_rate=0.05, subsample=0.8,
              colsample_bytree=0.8, objective='multi:softprob',
              num_class=len(le.classes_), eval_metric='mlogloss',
              tree_method='hist', device=XGB_DEVICE, random_state=SEED, n_jobs=-1)
models = {
    'depth3': XGBClassifier(max_depth=3, **common),
    'depth5': XGBClassifier(max_depth=5, **common),
}
# 위 설정은 비교용 예시입니다. 팀장님이 지정한 기존 설정과 다르면 반드시 조정하세요.
results = {}
for name, model in models.items():
    print('\n###', name)
    results[name] = run_cv(model, X_E, y, folds=folds)

summary = pd.DataFrame([{'model': k, 'cv_mean': v['mean'], 'cv_std': v['std'],
                         'oof_macro_f1': macro_f1(y, v['oof'])} for k,v in results.items()])
display(summary.sort_values('cv_mean', ascending=False))

Fold 1: train=4961, val=1240
Fold 2: train=4960, val=1241
Fold 3: train=4961, val=1240
Fold 4: train=4961, val=1240
Fold 5: train=4961, val=1240

### depth3


c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [17:28:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


fold 1: Macro F1 = 0.4754
fold 2: Macro F1 = 0.4962
fold 3: Macro F1 = 0.4612
fold 4: Macro F1 = 0.5019
fold 5: Macro F1 = 0.5001
CV Macro F1 = 0.4870 (std 0.0160)

### depth5
fold 1: Macro F1 = 0.4677
fold 2: Macro F1 = 0.4765
fold 3: Macro F1 = 0.4633
fold 4: Macro F1 = 0.4919
fold 5: Macro F1 = 0.4984
CV Macro F1 = 0.4796 (std 0.0136)


,model,cv_mean,cv_std,oof_macro_f1
0,depth3,0.486961,0.015966,0.488530
1,depth5,0.479570,0.013588,0.481739


In [6]:
# 모델 선택: 평균 Fold Macro F1 기준 (OOF 선택 편향에 유의)
best_name = max(results, key=lambda k: results[k]['mean'])
cv = results[best_name]
print('선택 모델:', best_name, 'CV:', cv['mean'], 'std:', cv['std'])
print('OOF 확률:', cv['oof_proba'].shape)
assert cv['oof_proba'].shape == (len(y), len(le.classes_))
assert np.isfinite(cv['oof_proba']).all()
assert np.allclose(cv['oof_proba'].sum(axis=1), 1.0, atol=1e-4)
display(per_class_f1(y, cv['oof'], le.classes_).head(10))

선택 모델: depth3 CV: 0.48696106620105206 std: 0.015965882126138378
OOF 확률: (6201, 26)


,f1-score,support
SARC,0.159696,198.0
PCPG,0.305344,147.0
KIRC,0.306796,334.0
THYM,0.314050,98.0
PAAD,0.331551,120.0
PRAD,0.349057,266.0
HNSC,0.390746,223.0
LUAD,0.401316,184.0
KIPAN,0.411067,515.0
DLBC,0.416667,38.0


In [7]:
# 최종 모델을 전체 train으로 1회 학습한 뒤 test 확률 생성
final_model = clone(models[best_name])
final_model.fit(X_E, y)
assert np.array_equal(final_model.classes_, np.arange(len(le.classes_)))
test_proba = final_model.predict_proba(X_test_E)
assert test_proba.shape == (len(test), len(le.classes_))
assert np.isfinite(test_proba).all()
assert np.allclose(test_proba.sum(axis=1), 1.0, atol=1e-4)
print('test 확률:', test_proba.shape)

test 확률: (2546, 26)


In [8]:
# 저장 전에 결과를 확인한 후 이 셀만 실행하세요.
experiment_name = f'xgb_E_group_1109_{best_name}'
log_result(owner='A_model', notebook='04_xgb_feature_E.ipynb',
           experiment=experiment_name, features=f'E_{X_E.shape[1]}',
           model='XGBClassifier', params=models[best_name].get_params(), cv=cv,
           notes='Group CV; ①②③ + binary + log mutation count; selected by CV')
save_predictions(owner='A_model', name=experiment_name,
                 oof_proba=cv['oof_proba'], test_proba=test_proba,
                 classes=le.classes_, cv=cv,
                 notes='Group CV; same train/test order as load_data; full-train test probabilities')
print('완료: experiments/log_A_model.csv 및 experiments/preds/')

기록: log_A_model.csv ← xgb_E_group_1109_depth3 (CV 0.4870)
예측 저장: A_model__xgb_E_group_1109_depth3 (CV 0.4870) → experiments/preds/
완료: experiments/log_A_model.csv 및 experiments/preds/


## PR 전 확인

1. `src/features.py` 최신 함수 네 개가 존재하는지 확인
2. 피처 열 수 **9,174개**, train/test 열 이름과 순서 일치
3. 5개 Fold의 검증 행이 누락·중복 없이 전체 train을 덮는지 확인
4. CV 결과와 OOF Macro F1 기록
5. 확률 배열과 클래스 순서 확인 후 저장
6. **본인 노트북과 본인 실험 산출물만** Git에 추가

**주의:** 이 노트북의 `common` XGBoost 파라미터는 비교 실험을 위한 예시입니다. 11/08 LB 0.3130 재현 설정과 동일하다고 주장하지 않습니다. 기존 제출 재현 노트북은 별도로 유지하세요.
